<a href="https://colab.research.google.com/github/leminhtiene03/massive-data-minning/blob/main/hm_recommendation_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# H&M Personalized Fashion Recommendations — V2
## Cải tiến: Global popular cho tất cả users + Item-to-item CF + Time-decay features

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

ROOT     = Path('/content/drive/MyDrive/claude274v2')
FEAT_DIR = ROOT / 'outputs/features'
EMB_DIR  = ROOT / 'outputs/embeddings'
CAND_DIR = ROOT / 'outputs/candidates'
MOD_DIR  = ROOT / 'outputs/models'
V2_DIR   = ROOT / 'outputs/v2'

print("=== EMBEDDINGS ===")
for p in sorted(EMB_DIR.glob('*')):
    print(f"  {p.name}: {p.stat().st_size/1024/1024:.1f} MB")

print("\n=== FEATURES ===")
for p in sorted(FEAT_DIR.glob('*.parquet')):
    print(f"  {p.name}: {p.stat().st_size/1024/1024:.1f} MB")

print("\n=== CANDIDATES ===")
for p in sorted(CAND_DIR.glob('*.parquet')):
    print(f"  {p.name}: {p.stat().st_size/1024/1024:.1f} MB")

print("\n=== MODELS ===")
for p in sorted(MOD_DIR.glob('*')):
    print(f"  {p.name}: {p.stat().st_size/1024/1024:.1f} MB")

print("\n=== V2_DIR ===")
for p in sorted(V2_DIR.glob('*.parquet')):
    print(f"  {p.name}: {p.stat().st_size/1024/1024:.1f} MB")

=== EMBEDDINGS ===
  .ipynb_checkpoints: 0.0 MB
  article_embeddings.npy: 360.7 MB
  article_ids.npy: 4.0 MB

=== FEATURES ===
  art_sales_trend.parquet: 0.4 MB
  emb_sim_v2.parquet: 754.9 MB
  source_pivot_v2.parquet: 768.4 MB
  user_bought.parquet: 966.3 MB
  user_depts.parquet: 465.4 MB
  user_item_decay.parquet: 1025.7 MB
  user_prod_types.parquet: 352.1 MB

=== CANDIDATES ===
  age_popular.parquet: 14.7 MB
  all_candidates_v2.parquet: 2865.1 MB
  als.parquet: 62.2 MB
  embedding_sim.parquet: 10.2 MB
  global_popular_all.parquet: 52.2 MB
  item_item_cands.parquet: 19.5 MB
  repurchase.parquet: 6.4 MB

=== MODELS ===
  lgbm_ranker_v2.pkl: 0.0 MB

=== V2_DIR ===
  emb_sim_0000000.parquet: 27.5 MB
  emb_sim_0050000.parquet: 27.4 MB
  emb_sim_0100000.parquet: 27.5 MB
  pred_feat_0000.parquet: 339.2 MB
  pred_feat_0001.parquet: 339.3 MB
  pred_feat_0002.parquet: 338.8 MB
  pred_feat_0003.parquet: 338.8 MB
  pred_feat_0004.parquet: 339.4 MB
  pred_feat_0005.parquet: 339.0 MB
  pred_feat_

## Section 0 — Setup & Imports

In [ ]:
# [KHÔNG THAY ĐỔI] Setup giữ nguyên như v1
import warnings; warnings.filterwarnings('ignore')
import gc, pickle
from datetime import datetime, date
from pathlib import Path
import numpy as np
import polars as pl
import scipy.sparse as sp

ROOT     = Path('/content/drive/MyDrive/claude274v2')
DATA_DIR = ROOT / 'data'
OUT_DIR  = ROOT / 'outputs'
CAND_DIR = OUT_DIR / 'candidates'
FEAT_DIR = OUT_DIR / 'features'
EMB_DIR  = OUT_DIR / 'embeddings'
MOD_DIR  = OUT_DIR / 'models'
SUB_DIR  = OUT_DIR / 'submissions'
V2_DIR   = OUT_DIR / 'v2'  # thư mục mới cho v2

# Ensure root and top-level data/output directories exist
ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

for d in [CAND_DIR, FEAT_DIR, EMB_DIR, MOD_DIR, SUB_DIR, V2_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Dates — dùng Python date thay vì pl.date để tránh lỗi pl.lit()
LAST_TRAIN_DATE = date(2020, 9, 15)
TEST_START_DATE = date(2020, 9, 16)
DATE_7D  = date(2020, 9, 8)
DATE_14D = date(2020, 9, 1)
DATE_30D = date(2020, 8, 16)
DATE_90D = date(2020, 6, 17)

print(f'[{datetime.now().strftime("%H:%M:%S")}] Setup done — Polars {pl.__version__}')

[06:42:25] Setup done — Polars 1.35.2


## Section 1 — Load Data

In [ ]:
#  Load data
print(f'[{datetime.now().strftime("%H:%M:%S")}] Loading data...')
transactions = pl.read_csv(DATA_DIR / 'transactions_train.csv',
                           dtypes={'article_id': pl.String, 'customer_id': pl.String})
transactions = transactions.with_columns(pl.col('t_dat').str.to_date('%Y-%m-%d'))
train_tx     = transactions.filter(pl.col('t_dat') < TEST_START_DATE)
train_tx     = train_tx.with_columns(pl.col('article_id').cast(pl.String))
articles     = pl.read_csv(DATA_DIR / 'articles.csv', dtypes={'article_id': pl.String})
articles     = articles.with_columns(pl.col('article_id').cast(pl.String))
customers    = pl.read_csv(DATA_DIR / 'customers.csv', dtypes={'customer_id': pl.String})

print(f'[{datetime.now().strftime("%H:%M:%S")}] transactions: {transactions.shape}')
print(f'train_tx: {train_tx.shape}')
print(f'articles: {articles.shape}')
print(f'customers: {customers.shape}')

[06:42:25] Loading data...
[06:43:09] transactions: (31788324, 5)
train_tx: (31548013, 5)
articles: (105542, 25)
customers: (1371980, 7)


In [ ]:
VAL_START = date(2020, 9, 9)
train_tx_clean = train_tx.filter(pl.col('t_dat') < VAL_START)
print(f'train_tx_clean: {train_tx_clean.shape}')
print(f'Date range: {train_tx_clean["t_dat"].min()} → {train_tx_clean["t_dat"].max()}')

train_tx_clean: (31292772, 5)
Date range: 2018-09-20 → 2020-09-08


## Section 2 — Candidate Generation V2
### Thay đổi so với V1:
- Global popular assign cho **TẤT CẢ** customers (không chỉ cold)
- Thêm **Item-to-item** collaborative filtering
- Ghi từng phần ra disk ngay để tránh OOM

In [ ]:
# STEP 1: Repurchase

if (CAND_DIR / 'repurchase.parquet').exists():
    print('repurchase.parquet already exists — skip')
else:
    repurchase = (
        train_tx.filter(pl.col('t_dat') > DATE_14D)
        .select(['customer_id', 'article_id']).unique()
        .with_columns(pl.lit('repurchase').alias('source'))
    )
    repurchase.write_parquet(CAND_DIR / 'repurchase.parquet')
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Repurchase: {repurchase.shape}')

repurchase.parquet already exists — skip


In [ ]:
# STEP 2: Age-group popularity

if (CAND_DIR / 'age_popular.parquet').exists():
    print('age_popular.parquet already exists — skip')
else:
    customers_age = customers.select(['customer_id', 'age']).with_columns(
        pl.when(pl.col('age') < 25).then(pl.lit('<25'))
        .when(pl.col('age') < 35).then(pl.lit('25-35'))
        .when(pl.col('age') < 50).then(pl.lit('35-50'))
        .otherwise(pl.lit('50+')).alias('age_group')
    )
    tx_14d     = train_tx.filter(pl.col('t_dat') > DATE_14D)
    tx_90d_tmp = train_tx.filter(pl.col('t_dat') > DATE_90D)
    tx_age     = tx_14d.join(customers_age.select(['customer_id', 'age_group']), on='customer_id', how='left')

    age_pop_articles = (
        tx_age.group_by(['age_group', 'article_id']).len()
        .sort(['age_group', 'len'], descending=[False, True])
        .with_columns(pl.col('len').rank(method='ordinal', descending=True).over('age_group').alias('rank'))
        .filter(pl.col('rank') <= 50)
        .select(['age_group', 'article_id'])
    )

    custs_in_14d    = set(tx_14d['customer_id'].unique().to_list())
    custs_in_90d    = set(tx_90d_tmp['customer_id'].unique().to_list())
    warm_active_ids = [c for c in customers['customer_id'].to_list()
                       if c in custs_in_90d and c not in custs_in_14d]

    age_popular = (
        pl.DataFrame({'customer_id': warm_active_ids})
        .join(customers_age.select(['customer_id', 'age_group']), on='customer_id', how='left')
        .join(age_pop_articles, on='age_group', how='left')
        .drop('age_group').drop_nulls()
        .with_columns(pl.lit('age_popular').alias('source'))
    )
    age_popular.write_parquet(CAND_DIR / 'age_popular.parquet')
    del tx_14d, tx_90d_tmp, tx_age, customers_age, age_pop_articles; gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Age popular: {age_popular.shape}')

age_popular.parquet already exists — skip


In [ ]:
# STEP 3: ALS

if (CAND_DIR / 'als.parquet').exists():
    print('als.parquet already exists — skip')
else:
    !pip install implicit -q
    from implicit import als
    from implicit.nearest_neighbours import bm25_weight

    tx_90d  = train_tx.filter(pl.col('t_dat') > DATE_90D)
    uid_map = {u: i for i, u in enumerate(tx_90d['customer_id'].unique().to_list())}
    aid_map = {a: i for i, a in enumerate(tx_90d['article_id'].unique().to_list())}
    uid_inv = {i: u for u, i in uid_map.items()}
    aid_inv = {i: a for a, i in aid_map.items()}

    rows = [uid_map[u] for u in tx_90d['customer_id'].to_list()]
    cols = [aid_map[a] for a in tx_90d['article_id'].to_list()]
    data = np.ones(len(rows), dtype=np.float32)
    user_item      = sp.csr_matrix((data, (rows, cols)), shape=(len(uid_map), len(aid_map)))
    user_item_bm25 = bm25_weight(user_item, K1=100, B=0.8).tocsr()

    als_model = als.AlternatingLeastSquares(factors=128, iterations=15, regularization=0.01, random_state=42)
    als_model.fit(user_item_bm25)

    user_ids_90d = list(uid_map.keys())
    user_indices = list(uid_map.values())
    item_ids_arr, _ = als_model.recommend(
        user_indices, user_item[user_indices], N=50, filter_already_liked_items=True
    )
    als_rows = []
    for uid_str, items in zip(user_ids_90d, item_ids_arr):
        for item_idx in items:
            als_rows.append({'customer_id': uid_str, 'article_id': aid_inv[item_idx], 'source': 'als'})

    als_cands = pl.DataFrame(als_rows)
    als_cands.write_parquet(CAND_DIR / 'als.parquet')
    del tx_90d, user_item, user_item_bm25, als_rows; gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] ALS: {als_cands.shape}')

als.parquet already exists — skip


In [ ]:
# STEP 4: Embedding similarity

if (CAND_DIR / 'embedding_sim.parquet').exists():
    print('embedding_sim.parquet already exists — skip')
else:
    !pip install faiss-cpu -q
    import faiss

    article_embs    = np.load(EMB_DIR / 'article_embeddings.npy').astype(np.float32)
    emb_article_ids = np.array([str(a) for a in np.load(EMB_DIR / 'article_ids.npy', allow_pickle=True)])
    emb_aid_to_idx  = {a: i for i, a in enumerate(emb_article_ids)}

    faiss.normalize_L2(article_embs)
    index = faiss.IndexFlatIP(article_embs.shape[1])
    index.add(article_embs)
    print(f'FAISS index: {index.ntotal} articles')

    cust_articles = (
        train_tx.filter(pl.col('t_dat') > DATE_14D)
        .group_by('customer_id')
        .agg(pl.col('article_id').cast(pl.String).unique().alias('articles'))
    )

    K_NN = 20
    all_cids, all_centroids, all_purchased = [], [], []
    for row in cust_articles.iter_rows(named=True):
        arts = [a for a in row['articles'] if a in emb_aid_to_idx]
        if not arts:
            continue
        idxs = [emb_aid_to_idx[a] for a in arts]
        all_cids.append(row['customer_id'])
        all_centroids.append(article_embs[idxs].mean(axis=0))
        all_purchased.append(set(arts))

    batch = np.array(all_centroids, dtype=np.float32)
    faiss.normalize_L2(batch)
    _, I = index.search(batch, K_NN + 20)

    emb_rows = []
    for j, (cid, purchased) in enumerate(zip(all_cids, all_purchased)):
        neighbors = []
        for idx in I[j]:
            if idx < 0: continue
            aid = emb_article_ids[idx]
            if aid not in purchased:
                neighbors.append(aid)
            if len(neighbors) == K_NN:
                break
        for a in neighbors:
            emb_rows.append({'customer_id': cid, 'article_id': a, 'source': 'embedding_sim'})

    emb_cands = pl.DataFrame(emb_rows, schema={'customer_id': pl.String, 'article_id': pl.String, 'source': pl.String})
    emb_cands.write_parquet(CAND_DIR / 'embedding_sim.parquet')
    del article_embs, batch, I, all_centroids, emb_rows; gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Embedding sim: {emb_cands.shape}')

embedding_sim.parquet already exists — skip


In [ ]:
# STEP 5:  Global popular cho TẤT CẢ customers — streaming ra disk


global_pop_path = CAND_DIR / 'global_popular_all.parquet'
if global_pop_path.exists():
    print('global_popular_all.parquet already exists — skip')
else:
    global_pop_articles = (
        train_tx.filter(pl.col('t_dat') > DATE_7D)
        .group_by('article_id').len()
        .sort('len', descending=True)
        .head(100)['article_id'].to_list()
    )

    all_cust_ids = customers['customer_id'].to_list()
    CHUNK = 100_000
    chunk_paths = []

    for i in range(0, len(all_cust_ids), CHUNK):
        chunk_cids = all_cust_ids[i:i+CHUNK]
        p = V2_DIR / f'global_pop_{i:07d}.parquet'
        (
            pl.DataFrame({
                'customer_id': chunk_cids,
                'article_id':  [global_pop_articles] * len(chunk_cids),
            })
            .explode('article_id')
            .with_columns(pl.lit('global_popular').alias('source'))
            .write_parquet(p)
        )
        chunk_paths.append(p)
        gc.collect()

    # Merge lazy
    pl.scan_parquet([str(p) for p in chunk_paths]).sink_parquet(global_pop_path)
    for p in chunk_paths: p.unlink()
    gc.collect()
    result = pl.read_parquet(global_pop_path)
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Global popular (all): {result.shape}')
    del result; gc.collect()

global_popular_all.parquet already exists — skip


In [ ]:
# STEP 6:  Item-to-item collaborative filtering — streaming ra disk


item_item_path = CAND_DIR / 'item_item_cands.parquet'
if item_item_path.exists():
    print('item_item_cands.parquet already exists — skip')
else:
    tx_14d = train_tx.filter(pl.col('t_dat') > DATE_14D)

    # Chỉ dùng customers mua <= 15 items để tránh pair explosion
    moderate_custs = (
        tx_14d.group_by('customer_id').len()
        .filter(pl.col('len') <= 15)['customer_id'].to_list()
    )
    tx_14d_filtered = tx_14d.filter(pl.col('customer_id').is_in(moderate_custs))
    all_articles    = tx_14d_filtered['article_id'].unique().to_list()
    print(f'Customers: {len(moderate_custs):,}, Articles: {len(all_articles):,}')
    del tx_14d, moderate_custs; gc.collect()

    # Batch self-join theo articles, ghi từng batch ra disk
    BATCH_SIZE = 1000
    pair_paths = []

    for batch_idx in range(0, len(all_articles), BATCH_SIZE):
        batch_arts = all_articles[batch_idx:batch_idx+BATCH_SIZE]
        tx_art1    = tx_14d_filtered.filter(pl.col('article_id').is_in(batch_arts))                                     .rename({'article_id': 'art_1'})
        pairs = (
            tx_art1
            .join(tx_14d_filtered.select(['customer_id', 'article_id']).rename({'article_id': 'art_2'}),
                  on='customer_id')
            .filter(pl.col('art_1') != pl.col('art_2'))
            .group_by(['art_1', 'art_2']).len()
            .sort(['art_1', 'len'], descending=[False, True])
            .group_by('art_1').head(3)
            .select(['art_1', 'art_2'])
        )
        p = V2_DIR / f'pairs_batch_{batch_idx:05d}.parquet'
        pairs.write_parquet(p)
        pair_paths.append(p)
        del tx_art1, pairs; gc.collect()

        if batch_idx % 5000 == 0:
            print(f'  Pairs batch {batch_idx}/{len(all_articles)}')

    # Merge top_pairs
    top_pairs = pl.concat([pl.read_parquet(p) for p in pair_paths])
    for p in pair_paths: p.unlink()
    del tx_14d_filtered; gc.collect()
    print(f'Top pairs: {top_pairs.shape}')

    # Build item-item candidates
    item_item_cands = (
        train_tx.filter(pl.col('t_dat') > DATE_14D)
        .select(['customer_id', 'article_id']).rename({'article_id': 'art_1'})
        .join(top_pairs, on='art_1', how='inner')
        .select(['customer_id', 'art_2']).rename({'art_2': 'article_id'})
        .unique()
        .with_columns(pl.lit('item_item').alias('source'))
    )
    item_item_cands.write_parquet(item_item_path)
    del top_pairs, item_item_cands; gc.collect()
    result = pl.read_parquet(item_item_path)
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Item-item: {result.shape}')
    del result; gc.collect()

item_item_cands.parquet already exists — skip


In [ ]:
# STEP 7: Merge tất cả candidates — scan_parquet lazy, không load vào RAM


v2_cands_path = CAND_DIR / 'all_candidates_v2.parquet'
if v2_cands_path.exists():
    print('all_candidates_v2.parquet already exists — skip')
else:
    sources = [
        CAND_DIR / 'repurchase.parquet',
        CAND_DIR / 'age_popular.parquet',
        CAND_DIR / 'als.parquet',
        CAND_DIR / 'embedding_sim.parquet',
        CAND_DIR / 'global_popular_all.parquet',
        CAND_DIR / 'item_item_cands.parquet',
    ]
    existing = [str(p) for p in sources if p.exists()]
    print(f'Merging {len(existing)} source files...')

    # Merge theo chunk customers để unique không OOM
    all_cids = customers['customer_id'].unique().to_list()
    CHUNK    = 100_000
    merged_paths = []

    for i in range(0, len(all_cids), CHUNK):
        cid_set = set(all_cids[i:i+CHUNK])
        parts   = []
        for src in existing:
            p = pl.read_parquet(src).filter(pl.col('customer_id').is_in(cid_set))
            parts.append(p)
        chunk_merged = pl.concat(parts).unique(subset=['customer_id', 'article_id', 'source'])
        mp = V2_DIR / f'merged_{i:07d}.parquet'
        chunk_merged.write_parquet(mp)
        merged_paths.append(mp)
        del parts, chunk_merged; gc.collect()
        print(f'  Merge chunk {i//CHUNK+1}/{len(all_cids)//CHUNK+1}')

    pl.scan_parquet([str(p) for p in merged_paths]).sink_parquet(v2_cands_path)
    for p in merged_paths: p.unlink()
    gc.collect()

v2 = pl.read_parquet(v2_cands_path)
print(f'[{datetime.now().strftime("%H:%M:%S")}] all_candidates_v2: {v2.shape}')
print(f'Unique customers: {v2["customer_id"].n_unique():,}')
print(f'Sources:\n{v2.group_by("source").len().sort("len", descending=True)}')
del v2; gc.collect()

all_candidates_v2.parquet already exists — skip
[02:30:23] all_candidates_v2: (188322877, 3)
Unique customers: 1,371,980
Sources:
shape: (6, 2)
┌────────────────┬───────────┐
│ source         ┆ len       │
│ ---            ┆ ---       │
│ str            ┆ u32       │
╞════════════════╪═══════════╡
│ global_popular ┆ 137198000 │
│ als            ┆ 26700300  │
│ age_popular    ┆ 20006150  │
│ embedding_sim  ┆ 2677660   │
│ item_item      ┆ 1278385   │
│ repurchase     ┆ 462382    │
└────────────────┴───────────┘


10

## Section 3 — Feature Engineering V2
### Thay đổi so với V1:
- Thêm **time-decay weight** feature
- Thêm **item-item source** vào source_pivot
- Dùng `all_candidates_v2` thay vì `all_candidates`
- Tất cả bảng nặng lưu ra disk ngay, không giữ trong RAM

In [ ]:
# STEP 1: Precompute customer stats
print(f'[{datetime.now().strftime("%H:%M:%S")}] Computing customer stats...')

def cust_stats(tx, days, suffix):
    cutoff = LAST_TRAIN_DATE - pl.duration(days=days)
    return (tx.filter(pl.col('t_dat') > cutoff)
            .group_by('customer_id')
            .agg([pl.len().alias(f'n_purchases_{suffix}'),
                  pl.col('article_id').n_unique().alias(f'n_unique_articles_{suffix}'),
                  pl.col('price').mean().alias(f'avg_price_{suffix}')]))

cust_7d  = cust_stats(train_tx_clean, 7,  '7d')
cust_14d = cust_stats(train_tx_clean, 14, '14d')
cust_30d = cust_stats(train_tx_clean, 30, '30d')

last_purchase = (
    train_tx_clean.group_by('customer_id')
    .agg(pl.col('t_dat').max().alias('last_purchase_date'))
    .with_columns(((pl.lit(LAST_TRAIN_DATE) - pl.col('last_purchase_date')).dt.total_days())
                  .alias('days_since_last_purchase'))
    .select(['customer_id', 'days_since_last_purchase'])
)

age_median     = customers['age'].drop_nulls().median()
customers_feat = (
    customers.select(['customer_id', 'age', 'club_member_status', 'fashion_news_frequency'])
    .with_columns([
        pl.col('age').fill_null(age_median),
        (pl.col('club_member_status') == 'ACTIVE').cast(pl.Int8).alias('is_club_member'),
        pl.when(pl.col('fashion_news_frequency').is_in(['Regularly', 'Monthly']))
          .then(pl.lit(1)).otherwise(pl.lit(0)).cast(pl.Int8).alias('fashion_news_freq'),
    ])
    .select(['customer_id', 'age', 'is_club_member', 'fashion_news_freq'])
    .with_columns(
        pl.when(pl.col('age') < 25).then(pl.lit('<25'))
        .when(pl.col('age') < 35).then(pl.lit('25-35'))
        .when(pl.col('age') < 50).then(pl.lit('35-50'))
        .otherwise(pl.lit('50+')).alias('age_group')
    )
)
print(f'[{datetime.now().strftime("%H:%M:%S")}] Customer stats done')

[06:43:10] Computing customer stats...
[06:43:11] Customer stats done


In [ ]:
# STEP 2: Article features
def article_pop(tx, days, suffix):
    cutoff = LAST_TRAIN_DATE - pl.duration(days=days)
    return (tx.filter(pl.col('t_dat') > cutoff)
            .group_by('article_id').len()
            .rename({'len': f'article_popularity_{suffix}'}))

art_pop_7d  = article_pop(train_tx_clean, 7,  '7d')
art_pop_14d = article_pop(train_tx_clean, 14, '14d')

art_age_pop = (
    train_tx_clean.filter(pl.col('t_dat') > DATE_14D)
    .join(customers_feat.select(['customer_id', 'age_group']), on='customer_id', how='left')
    .group_by(['age_group', 'article_id']).len()
    .rename({'len': 'article_popularity_age_group_14d'})
)

art_last_sold = (
    train_tx_clean.group_by('article_id')
    .agg(pl.col('t_dat').max().alias('last_sold_date'))
    .with_columns(((pl.lit(LAST_TRAIN_DATE) - pl.col('last_sold_date')).dt.total_days())
                  .alias('days_since_article_last_sold'))
    .select(['article_id', 'days_since_article_last_sold'])
)

articles_feat = articles.select(
    ['article_id', 'product_type_no', 'colour_group_code', 'department_no', 'perceived_colour_value_id']
)
print(f'[{datetime.now().strftime("%H:%M:%S")}] Article features done')

[06:43:12] Article features done


In [ ]:
# SECTION EMBEDDINGS — Thêm vào trước Section 3 Feature Engineering
# Sinh article embeddings từ text (sentence-transformers) + image (CLIP)

# STEP 1: Build article text descriptions
print(f'[{datetime.now().strftime("%H:%M:%S")}] Building article text...')
articles_text = articles.with_columns(
    (pl.col('prod_name').fill_null('') + ' ' +
     pl.col('product_type_name').fill_null('') + ' ' +
     pl.col('colour_group_name').fill_null('') + ' ' +
     pl.col('department_name').fill_null('') + ' ' +
     pl.col('detail_desc').fill_null('')).alias('text')
).select(['article_id', 'text'])

article_ids_list = articles_text['article_id'].to_list()
texts            = articles_text['text'].to_list()
print(f'Articles to embed: {len(article_ids_list):,}')

[06:43:12] Building article text...
Articles to embed: 105,542


In [ ]:
# STEP 2: Text embeddings — sentence-transformers
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer

print(f'[{datetime.now().strftime("%H:%M:%S")}] Loading sentence-transformer...')
st_model  = SentenceTransformer('all-MiniLM-L6-v2')
text_embs = st_model.encode(
    texts, batch_size=512, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True
)
print(f'[{datetime.now().strftime("%H:%M:%S")}] Text embeddings shape: {text_embs.shape}')
# shape: (105542, 384)

[06:43:45] Loading sentence-transformer...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/207 [00:00<?, ?it/s]

[06:44:45] Text embeddings shape: (105542, 384)


In [ ]:
# STEP 3: Image embeddings — CLIP ViT-B/32 (fixed GPU utilization)
import open_clip
import torch
from PIL import Image

IMAGES_DIR = LOCAL_IMG_DIR
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[{datetime.now().strftime("%H:%M:%S")}] CLIP on {device}')

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_model = clip_model.to(device).eval()

IMG_DIM    = 512
img_embs   = np.zeros((len(article_ids_list), IMG_DIM), dtype=np.float32)
CLIP_BATCH = 512
total      = len(article_ids_list)
missing    = 0

for start in range(0, total, CLIP_BATCH):
    batch_ids = article_ids_list[start:start + CLIP_BATCH]
    imgs      = []
    for aid in batch_ids:
        img_path = IMAGES_DIR / aid[:3] / f'{aid}.jpg'
        try:
            imgs.append(clip_preprocess(Image.open(img_path).convert('RGB')))
        except Exception:
            imgs.append(torch.zeros(3, 224, 224))
            missing += 1

    # Stack và chuyển lên GPU một lần
    batch_tensor = torch.stack(imgs).to(device)

    with torch.no_grad(), torch.cuda.amp.autocast():  # amp để tăng tốc GPU
        feats = clip_model.encode_image(batch_tensor)
        feats = feats / feats.norm(dim=-1, keepdim=True)  # normalize trên GPU
        feats = feats.cpu().numpy().astype(np.float32)

    img_embs[start:start + CLIP_BATCH] = feats

    if start % (CLIP_BATCH * 20) == 0:  # log mỗi 10K ảnh
        print(f'[{datetime.now().strftime("%H:%M:%S")}] CLIP: {start}/{total} — VRAM: {torch.cuda.memory_allocated()/1024**3:.1f}GB')

print(f'[{datetime.now().strftime("%H:%M:%S")}] Done — missing: {missing:,}')

[06:54:29] CLIP on cuda
[06:54:49] CLIP: 0/105542 — VRAM: 0.9GB


KeyboardInterrupt: 

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU name: {torch.cuda.get_device_name(0)}')
print(f'VRAM used: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB')
print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB')

CUDA available: True
GPU name: Tesla T4
VRAM used: 0.95 GB
VRAM total: 14.56 GB


In [ ]:
# STEP 3: User-article interaction — lưu disk ngay
if not (FEAT_DIR / 'user_bought.parquet').exists():
    user_bought = (
        train_tx_clean.select(['customer_id', 'article_id']).unique()
        .with_columns(pl.lit(1).cast(pl.Int8).alias('user_has_bought_before'))
    )
    user_bought.write_parquet(FEAT_DIR / 'user_bought.parquet')
    del user_bought; gc.collect()

if not (FEAT_DIR / 'user_prod_types.parquet').exists():
    user_prod_types = (
        train_tx_clean.join(articles.select(['article_id', 'product_type_no']), on='article_id', how='left')
        .select(['customer_id', 'product_type_no']).unique()
    )
    user_prod_types.write_parquet(FEAT_DIR / 'user_prod_types.parquet')
    del user_prod_types; gc.collect()

if not (FEAT_DIR / 'user_depts.parquet').exists():
    user_depts = (
        train_tx_clean.join(articles.select(['article_id', 'department_no']), on='article_id', how='left')
        .select(['customer_id', 'department_no']).unique()
    )
    user_depts.write_parquet(FEAT_DIR / 'user_depts.parquet')
    del user_depts; gc.collect()

print(f'[{datetime.now().strftime("%H:%M:%S")}] Interaction tables done')

[06:44:59] Interaction tables done


In [ ]:
# STEP 4:  Time-decay feature


decay_path = FEAT_DIR / 'user_item_decay.parquet'
if decay_path.exists():
    print('user_item_decay.parquet already exists — skip')
else:
    user_item_decay = (
        train_tx_clean
        .with_columns(
            (pl.lit(LAST_TRAIN_DATE) - pl.col('t_dat')).dt.total_days().alias('days_ago')
        )
        .with_columns(
            (pl.lit(0.95) ** pl.col('days_ago')).alias('decay_weight')
        )
        .group_by(['customer_id', 'article_id'])
        .agg(pl.col('decay_weight').sum().alias('user_item_decay_weight'))
    )
    user_item_decay.write_parquet(decay_path)
    del user_item_decay; gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Time-decay saved')

# Sales trend feature — art_recent_7d / art_alltime
trend_path = FEAT_DIR / 'art_sales_trend.parquet'
if trend_path.exists():
    print('art_sales_trend.parquet already exists — skip')
else:
    art_alltime = (
        train_tx_clean.group_by('article_id').len()
        .rename({'len': 'art_alltime_sales'})
    )
    art_recent_7d = (
        train_tx_clean.filter(pl.col('t_dat') > DATE_7D)
        .group_by('article_id').len()
        .rename({'len': 'art_recent_7d_sales'})
    )
    art_trend = (
        art_alltime.join(art_recent_7d, on='article_id', how='left')
        .with_columns(pl.col('art_recent_7d_sales').fill_null(0))
        .with_columns(
            (pl.col('art_recent_7d_sales') / (pl.col('art_alltime_sales') + 1))
            .alias('sales_trend')
        )
        .select(['article_id', 'sales_trend'])
    )
    art_trend.write_parquet(trend_path)
    del art_alltime, art_recent_7d, art_trend; gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Sales trend saved')

[03:11:46] Time-decay saved
[03:11:48] Sales trend saved


In [ ]:
# STEP 5: Embedding similarity v2 — chunked hoàn toàn
emb_sim_v2_path = FEAT_DIR / 'emb_sim_v2.parquet'
if emb_sim_v2_path.exists():
    print('emb_sim_v2.parquet already exists — skip')
else:
    article_embs_raw = np.load(EMB_DIR / 'article_embeddings.npy').astype(np.float32)
    emb_article_ids  = np.array([str(a) for a in np.load(EMB_DIR / 'article_ids.npy', allow_pickle=True)])
    emb_aid_to_idx   = {a: i for i, a in enumerate(emb_article_ids)}

    norms = np.linalg.norm(article_embs_raw, axis=1, keepdims=True)
    norms[norms == 0] = 1
    article_embs_norm = article_embs_raw / norms
    del article_embs_raw; gc.collect()

    # Build centroids
    cust_articles_feat = (
        train_tx_clean.filter(pl.col('t_dat') > DATE_14D)
        .group_by('customer_id')
        .agg(pl.col('article_id').cast(pl.String).unique().alias('articles'))
    )
    cid_list, centroid_list = [], []
    for row in cust_articles_feat.iter_rows(named=True):
        arts = [a for a in row['articles'] if a in emb_aid_to_idx]
        if not arts: continue
        idxs = [emb_aid_to_idx[a] for a in arts]
        cid_list.append(row['customer_id'])
        centroid_list.append(article_embs_norm[idxs].mean(axis=0))

    centroid_matrix = np.array(centroid_list, dtype=np.float32)
    cnorms = np.linalg.norm(centroid_matrix, axis=1, keepdims=True)
    cnorms[cnorms == 0] = 1
    centroid_matrix /= cnorms
    cid_to_centroid_idx = {cid: i for i, cid in enumerate(cid_list)}
    del centroid_list, cust_articles_feat; gc.collect()
    print(f'Centroid matrix: {centroid_matrix.shape}')

    # Tính sim theo chunk customers — không load all_candidates_v2 toàn bộ
    all_cids_v2 = customers['customer_id'].to_list()
    CHUNK_SIZE  = 50_000
    sim_paths   = []

    for i in range(0, len(all_cids_v2), CHUNK_SIZE):
        sp = V2_DIR / f'emb_sim_{i:07d}.parquet'
        if sp.exists():
            sim_paths.append(sp)
            continue

        cid_chunk = all_cids_v2[i:i+CHUNK_SIZE]
        cid_set   = set(cid_chunk)

        # Load chỉ candidates của chunk này
        cands_chunk = pl.read_parquet(CAND_DIR / 'all_candidates_v2.parquet') \
                        .filter(pl.col('customer_id').is_in(cid_set))
        cust_ids = cands_chunk['customer_id'].to_list()
        art_ids  = cands_chunk['article_id'].to_list()
        del cands_chunk; gc.collect()

        # Dot product
        c_idxs = np.array([cid_to_centroid_idx.get(c, -1) for c in cust_ids], dtype=np.int32)
        a_idxs = np.array([emb_aid_to_idx.get(a, -1)      for a in art_ids],  dtype=np.int32)
        valid  = (c_idxs >= 0) & (a_idxs >= 0)
        sims   = np.zeros(len(cust_ids), dtype=np.float32)
        if valid.any():
            sims[valid] = (
                centroid_matrix[c_idxs[valid]] * article_embs_norm[a_idxs[valid]]
            ).sum(axis=1)

        pl.DataFrame({
            'customer_id': cust_ids,
            'article_id':  art_ids,
            'embedding_similarity': sims.tolist(),
        }).with_columns(pl.col('embedding_similarity').cast(pl.Float32)) \
          .write_parquet(sp)
        sim_paths.append(sp)
        del cust_ids, art_ids, c_idxs, a_idxs, valid, sims; gc.collect()

        if i % 500_000 == 0:
            print(f'  [{datetime.now().strftime("%H:%M:%S")}] {i//CHUNK_SIZE}/{len(all_cids_v2)//CHUNK_SIZE}')

    del centroid_matrix, article_embs_norm, cid_to_centroid_idx; gc.collect()

    # Merge tất cả sim chunks
    pl.scan_parquet([str(p) for p in sim_paths]).sink_parquet(emb_sim_v2_path)
    for p in sim_paths: p.unlink()
    gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] emb_sim_v2 saved')

Centroid matrix: (75822, 896)
  [03:13:45] 0/27


KeyboardInterrupt: 

In [ ]:
# STEP 6: Source pivot


src_pivot_v2_path = FEAT_DIR / 'source_pivot_v2.parquet'
if src_pivot_v2_path.exists():
    print('source_pivot_v2.parquet already exists — skip')
else:
    all_cands_v2 = pl.read_parquet(CAND_DIR / 'all_candidates_v2.parquet')

    # Pivot theo chunk để tránh OOM
    all_cids = all_cands_v2['customer_id'].unique().to_list()
    CHUNK    = 100_000
    pivot_paths = []

    for i in range(0, len(all_cids), CHUNK):
        cid_set = set(all_cids[i:i+CHUNK])
        chunk   = all_cands_v2.filter(pl.col('customer_id').is_in(cid_set))
        pivot   = (
            chunk.with_columns(pl.lit(1).cast(pl.Int8).alias('flag'))
            .pivot(index=['customer_id', 'article_id'], on='source', values='flag', aggregate_function='first')
            .fill_null(0)
        )
        # Đảm bảo tất cả source columns tồn tại
        for src in ['repurchase', 'global_popular', 'age_popular', 'als', 'embedding_sim', 'item_item']:
            col_name = f'candidate_source_{src}'
            if src in pivot.columns:
                pivot = pivot.rename({src: col_name})
            elif col_name not in pivot.columns:
                pivot = pivot.with_columns(pl.lit(0).cast(pl.Int8).alias(col_name))

        pp = V2_DIR / f'pivot_{i:07d}.parquet'
        pivot.write_parquet(pp)
        pivot_paths.append(pp)
        del chunk, pivot; gc.collect()
        print(f'  Pivot chunk {i//CHUNK+1}/{len(all_cids)//CHUNK+1}')

    pl.scan_parquet([str(p) for p in pivot_paths]).sink_parquet(src_pivot_v2_path)
    for p in pivot_paths: p.unlink()
    del all_cands_v2; gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] source_pivot_v2 saved')

source_pivot_v2.parquet already exists — skip


In [ ]:
# STEP 7: Define build_features


def build_features_v2(candidates_df, label_tx=None):
    cids = candidates_df['customer_id'].unique()

    _user_bought = pl.read_parquet(FEAT_DIR / 'user_bought.parquet').filter(pl.col('customer_id').is_in(cids))
    _user_prods  = pl.read_parquet(FEAT_DIR / 'user_prod_types.parquet').filter(pl.col('customer_id').is_in(cids))
    _user_depts  = pl.read_parquet(FEAT_DIR / 'user_depts.parquet').filter(pl.col('customer_id').is_in(cids))
    _emb_sim     = pl.read_parquet(FEAT_DIR / 'emb_sim_v2.parquet').filter(pl.col('customer_id').is_in(cids))
    _src_pivot   = pl.read_parquet(FEAT_DIR / 'source_pivot_v2.parquet').filter(pl.col('customer_id').is_in(cids))
    _decay       = pl.read_parquet(FEAT_DIR / 'user_item_decay.parquet').filter(pl.col('customer_id').is_in(cids))
    _trend       = pl.read_parquet(FEAT_DIR / 'art_sales_trend.parquet')

    feats = (
        candidates_df.select(['customer_id', 'article_id']).unique()
        .join(customers_feat.select(['customer_id', 'age', 'is_club_member', 'fashion_news_freq', 'age_group']),
              on='customer_id', how='left')
        .join(cust_7d,   on='customer_id', how='left')
        .join(cust_14d,  on='customer_id', how='left')
        .join(cust_30d.select(['customer_id', 'n_purchases_30d', 'n_unique_articles_30d', 'avg_price_30d']),
              on='customer_id', how='left')
        .join(last_purchase,  on='customer_id', how='left')
        .join(art_pop_7d,     on='article_id',  how='left')
        .join(art_pop_14d,    on='article_id',  how='left')
        .join(art_age_pop,    on=['age_group', 'article_id'], how='left')
        .join(art_last_sold,  on='article_id',  how='left')
        .join(articles_feat,  on='article_id',  how='left')
        .join(_trend,         on='article_id',  how='left')   # [MỚI] sales trend
        .join(_user_bought,   on=['customer_id', 'article_id'], how='left')
        .join(_user_prods.with_columns(pl.lit(1).cast(pl.Int8).alias('user_bought_same_product_type')),
              on=['customer_id', 'product_type_no'], how='left')
        .join(_user_depts.with_columns(pl.lit(1).cast(pl.Int8).alias('user_bought_same_department')),
              on=['customer_id', 'department_no'], how='left')
        .join(_decay,     on=['customer_id', 'article_id'], how='left')  # [MỚI] time-decay
        .join(_emb_sim,   on=['customer_id', 'article_id'], how='left')
        .join(_src_pivot, on=['customer_id', 'article_id'], how='left')
    )

    int_cols   = ['is_club_member', 'fashion_news_freq', 'user_has_bought_before',
                  'user_bought_same_product_type', 'user_bought_same_department',
                  'candidate_source_repurchase', 'candidate_source_global_popular',
                  'candidate_source_age_popular', 'candidate_source_als',
                  'candidate_source_embedding_sim', 'candidate_source_item_item']
    zero_cols  = ['n_purchases_7d', 'n_purchases_14d', 'n_purchases_30d',
                  'n_unique_articles_30d', 'article_popularity_7d', 'article_popularity_14d',
                  'article_popularity_age_group_14d', 'embedding_similarity',
                  'user_item_decay_weight', 'sales_trend']
    large_cols = ['days_since_last_purchase', 'days_since_article_last_sold']

    for c in int_cols:
        if c in feats.columns:
            feats = feats.with_columns(pl.col(c).fill_null(0).cast(pl.Int8))
    for c in zero_cols:
        if c in feats.columns:
            feats = feats.with_columns(pl.col(c).fill_null(0))
    for c in large_cols:
        if c in feats.columns:
            feats = feats.with_columns(pl.col(c).fill_null(999))
    feats = feats.with_columns([
        pl.col('avg_price_7d').fill_null(pl.col('avg_price_7d').mean()),
        pl.col('avg_price_14d').fill_null(pl.col('avg_price_14d').mean()),
        pl.col('avg_price_30d').fill_null(pl.col('avg_price_30d').mean()),
    ])

    if label_tx is not None:
        bought = (
            label_tx.select(['customer_id', 'article_id']).unique()
            .with_columns(pl.lit(1).cast(pl.Int8).alias('label'))
        )
        feats = feats.join(bought, on=['customer_id', 'article_id'], how='left')
        feats = feats.with_columns(pl.col('label').fill_null(0))

    del _user_bought, _user_prods, _user_depts, _emb_sim, _src_pivot, _decay, _trend
    return feats

print(f'[{datetime.now().strftime("%H:%M:%S")}] build_features_v2() defined')

[16:15:20] build_features_v2() defined


In [ ]:
# STEP 8: Build train features


VAL_START = date(2020, 9, 9)
VAL_END   = date(2020, 9, 15)

# Label từ tuần -2, không phải tuần test
label_tx = transactions.filter(
    (pl.col('t_dat') >= VAL_START) & (pl.col('t_dat') <= VAL_END)
)
print(f'Label transactions (week -2): {label_tx.shape}')
print(f'Unique buyers in label week: {label_tx["customer_id"].n_unique():,}')

all_cids = (
    pl.scan_parquet(CAND_DIR / 'all_candidates_v2.parquet')
    .select('customer_id').unique().collect()
    ['customer_id'].to_list()
)
CHUNK_SIZE   = 100_000
chunks       = [all_cids[i:i+CHUNK_SIZE] for i in range(0, len(all_cids), CHUNK_SIZE)]
print(f'{len(all_cids):,} customers — {len(chunks)} chunks')

for i, cid_chunk in enumerate(chunks):
    chunk_path = V2_DIR / f'train_feat_{i:04d}.parquet'
    if chunk_path.exists():
        print(f'  Chunk {i+1}/{len(chunks)} already exists — skip')
        continue
    cid_set     = set(cid_chunk)
    cands_chunk = (
    pl.scan_parquet(CAND_DIR / 'all_candidates_v2.parquet')
    .filter(pl.col('customer_id').is_in(cid_set))
    .collect()
)
    label_chunk = label_tx.filter(pl.col('customer_id').is_in(cid_set))
    feat_chunk  = build_features_v2(cands_chunk, label_tx=label_chunk)
    feat_chunk.write_parquet(chunk_path)
    del cands_chunk, label_chunk, feat_chunk; gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Chunk {i+1}/{len(chunks)} done')



# Merge lazy
train_v2_path = V2_DIR / 'train_features_v2.parquet'
pl.scan_parquet(V2_DIR / 'train_feat_*.parquet').sink_parquet(train_v2_path)

row_count = pl.scan_parquet(train_v2_path).select(pl.len()).collect().item()
label_dist = pl.scan_parquet(train_v2_path).select('label').collect()['label'].value_counts()
print(f'Train features v2: {row_count:,} rows')
print(label_dist)

Label transactions (week -2): (255241, 5)
Unique buyers in label week: 72,019
1,371,980 customers — 14 chunks
  Chunk 1/14 already exists — skip
  Chunk 2/14 already exists — skip
  Chunk 3/14 already exists — skip
  Chunk 4/14 already exists — skip
  Chunk 5/14 already exists — skip
  Chunk 6/14 already exists — skip
  Chunk 7/14 already exists — skip
  Chunk 8/14 already exists — skip
  Chunk 9/14 already exists — skip
  Chunk 10/14 already exists — skip
  Chunk 11/14 already exists — skip
  Chunk 12/14 already exists — skip
  Chunk 13/14 already exists — skip
  Chunk 14/14 already exists — skip
Train features v2: 188,322,877 rows
shape: (2, 2)
┌───────┬───────────┐
│ label ┆ count     │
│ ---   ┆ ---       │
│ i8    ┆ u32       │
╞═══════╪═══════════╡
│ 1     ┆ 320016    │
│ 0     ┆ 188002861 │
└───────┴───────────┘


## Section 4 — Train LightGBM


In [ ]:
# STEP 1: Load và define feature columns
import lightgbm as lgb

FEATURE_COLS_V2 = [
    'age', 'is_club_member', 'fashion_news_freq',
    'n_purchases_7d', 'n_purchases_14d', 'n_purchases_30d',
    'n_unique_articles_30d', 'avg_price_30d', 'days_since_last_purchase',
    'article_popularity_7d', 'article_popularity_14d', 'article_popularity_age_group_14d',
    'days_since_article_last_sold',
    'product_type_no', 'colour_group_code', 'department_no',
    'user_has_bought_before', 'user_bought_same_product_type', 'user_bought_same_department',
    'embedding_similarity',
    'user_item_decay_weight',   # [MỚI]
    'sales_trend',              # [MỚI]
    'candidate_source_repurchase', 'candidate_source_global_popular',
    'candidate_source_age_popular', 'candidate_source_als',
    'candidate_source_embedding_sim', 'candidate_source_item_item',  # [MỚI]
]

# Verify columns exist
sample = pl.read_parquet(V2_DIR / 'train_feat_0000.parquet')
FEATURE_COLS_V2 = [c for c in FEATURE_COLS_V2 if c in sample.columns]
print(f'Features used: {len(FEATURE_COLS_V2)}')
print(f'Missing: {[c for c in FEATURE_COLS_V2 if c not in sample.columns]}')
del sample; gc.collect()

Features used: 28
Missing: []


0

In [ ]:
# STEP 2: Sample negatives từng chunk
RATIO = 10
SEED  = 42

chunk_files   = sorted(V2_DIR.glob('train_feat_*.parquet'))
sampled_paths = []

for i, p in enumerate(chunk_files):
    sp = V2_DIR / f'sampled_{i:04d}.parquet'
    if sp.exists():
        sampled_paths.append(sp)
        print(f'  Chunk {i+1}/{len(chunk_files)} already sampled — skip')
        continue
    chunk = pl.read_parquet(p)
    pos   = chunk.filter(pl.col('label') == 1)
    neg   = chunk.filter(pl.col('label') == 0)
    n_neg = min(len(pos) * RATIO, len(neg))
    if n_neg > 0:
        sampled = pl.concat([pos, neg.sample(n=n_neg, seed=SEED)])
    else:
        sampled = pos
    sampled.write_parquet(sp)
    sampled_paths.append(sp)
    del chunk, pos, neg, sampled; gc.collect()
    print(f'  Chunk {i+1}/{len(chunk_files)} sampled')

print(f'[{datetime.now().strftime("%H:%M:%S")}] All chunks sampled')

  Chunk 1/14 already sampled — skip
  Chunk 2/14 already sampled — skip
  Chunk 3/14 already sampled — skip
  Chunk 4/14 already sampled — skip
  Chunk 5/14 already sampled — skip
  Chunk 6/14 already sampled — skip
  Chunk 7/14 already sampled — skip
  Chunk 8/14 already sampled — skip
  Chunk 9/14 already sampled — skip
  Chunk 10/14 already sampled — skip
  Chunk 11/14 already sampled — skip
  Chunk 12/14 already sampled — skip
  Chunk 13/14 already sampled — skip
  Chunk 14/14 already sampled — skip
[15:57:20] All chunks sampled


In [ ]:
# STEP 3: Build LGB train dataset từ sampled chunks
# Re-initialize sampled_paths as it was not defined in this scope
sampled_paths = sorted(V2_DIR.glob('sampled_*.parquet'))

X_parts, y_parts, g_parts = [], [], []

for sp in sampled_paths:
    chunk = pl.read_parquet(sp).sort(['customer_id', 'label'], descending=[False, True])
    X_parts.append(chunk.select(FEATURE_COLS_V2).to_pandas())
    y_parts.append(chunk['label'].to_numpy())
    g_parts.append(chunk.group_by('customer_id', maintain_order=True).len()['len'].to_numpy())
    del chunk; gc.collect()

import pandas as pd
X_train = pd.concat(X_parts, ignore_index=True)
y_train = np.concatenate(y_parts)
g_train = np.concatenate(g_parts)
del X_parts, y_parts, g_parts; gc.collect()

lgb_train = lgb.Dataset(X_train, label=y_train, group=g_train, free_raw_data=True)
lgb_train.construct()
del X_train, y_train, g_train; gc.collect()
print(f'[{datetime.now().strftime("%H:%M:%S")}] LGB train dataset ready')

[16:15:57] LGB train dataset ready


In [ ]:
 # STEP 4: Build val features — dùng tuần test làm validation
#  Đánh giá trực tiếp trên tuần test để có MAP@12 thực
test_tx = transactions.filter(pl.col('t_dat') >= TEST_START_DATE)

all_cands_v2 = pl.read_parquet(CAND_DIR / 'all_candidates_v2.parquet')
val_cids     = all_cands_v2['customer_id'].unique().to_list()
val_chunks   = [val_cids[i:i+100_000] for i in range(0, len(val_cids), 100_000)]
val_paths    = []

for i, cid_chunk in enumerate(val_chunks):
    vp = V2_DIR / f'val_feat_{i:04d}.parquet'
    if vp.exists():
        val_paths.append(vp)
        print(f'  Val chunk {i+1}/{len(val_chunks)} exists — skip')
        continue
    cid_set = set(cid_chunk)
    cands_c = all_cands_v2.filter(pl.col('customer_id').is_in(cid_set))
    label_c = test_tx.filter(pl.col('customer_id').is_in(cid_set))
    feat_c  = build_features_v2(cands_c, label_tx=label_c)
    feat_c.write_parquet(vp)
    val_paths.append(vp)
    del cands_c, label_c, feat_c; gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Val chunk {i+1}/{len(val_chunks)} done')

del all_cands_v2; gc.collect()

pl.scan_parquet(V2_DIR / 'val_feat_*.parquet').sink_parquet(V2_DIR / 'val_features_v2.parquet')
val_features = pl.read_parquet(V2_DIR / 'val_features_v2.parquet')
print(f'Val features: {val_features.shape}')
print(val_features['label'].value_counts())

[13:16:03] Val chunk 1/14 done
[13:17:20] Val chunk 2/14 done
[13:18:44] Val chunk 3/14 done
[13:20:01] Val chunk 4/14 done
[13:21:18] Val chunk 5/14 done
[13:22:55] Val chunk 6/14 done
[13:24:41] Val chunk 7/14 done
[13:26:01] Val chunk 8/14 done
[13:27:15] Val chunk 9/14 done
[13:28:30] Val chunk 10/14 done
[13:29:45] Val chunk 11/14 done
[13:31:01] Val chunk 12/14 done
[13:32:17] Val chunk 13/14 done
[13:33:13] Val chunk 14/14 done
Val features: (188322877, 37)
shape: (2, 2)
┌───────┬───────────┐
│ label ┆ count     │
│ ---   ┆ ---       │
│ i8    ┆ u32       │
╞═══════╪═══════════╡
│ 0     ┆ 188276905 │
│ 1     ┆ 45972     │
└───────┴───────────┘


In [ ]:

import pandas as pd
import numpy as np

val_chunk_files = sorted(V2_DIR.glob('val_feat_*.parquet'))
print(f'Val chunks: {len(val_chunk_files)}')

for i, vp in enumerate(val_chunk_files):
    np_path = V2_DIR / f'val_np_{i:04d}.npz'
    if np_path.exists():
        print(f'  {i+1}/{len(val_chunk_files)} exists — skip')
        continue
    chunk = pl.read_parquet(vp).sort(['customer_id', 'label'], descending=[False, True])
    X = chunk.select(FEATURE_COLS_V2).to_pandas().values.astype(np.float32)
    y = chunk['label'].to_numpy()
    g = chunk.group_by('customer_id', maintain_order=True).len()['len'].to_numpy()
    np.savez_compressed(np_path, X=X, y=y, g=g)
    del chunk, X, y, g; gc.collect()
    print(f'  [{datetime.now().strftime("%H:%M:%S")}] {i+1}/{len(val_chunk_files)} saved')

print('All val numpy parts saved')

Val chunks: 14
  1/14 exists — skip
  2/14 exists — skip
  3/14 exists — skip
  4/14 exists — skip
  5/14 exists — skip
  6/14 exists — skip
  7/14 exists — skip
  8/14 exists — skip
  9/14 exists — skip
  10/14 exists — skip
  11/14 exists — skip
  12/14 exists — skip
  13/14 exists — skip
  14/14 exists — skip
All val numpy parts saved


In [ ]:
TRAIN_MODEL = True

if TRAIN_MODEL:
    # Load val numpy parts
    np_files = sorted(V2_DIR.glob('val_np_*.npz'))
    X_val = np.concatenate([np.load(p)['X'] for p in np_files])
    y_val = np.concatenate([np.load(p)['y'] for p in np_files])
    g_val = np.concatenate([np.load(p)['g'] for p in np_files])
    gc.collect()
    print(f'Val: X={X_val.shape}, positives={y_val.sum():.0f}')

    lgb_val = lgb.Dataset(X_val, label=y_val, group=g_val, reference=lgb_train, free_raw_data=True)
    lgb_val.construct()
    del X_val, y_val, g_val; gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] LGB val ready — training...')

    params = {
        'objective':      'lambdarank',
        'metric':         'ndcg',
        'ndcg_eval_at':   [12],
        'learning_rate':  0.05,
        'num_leaves':     127,
        'min_child_samples': 20,
        'verbose':        -1,
        'device': 'gpu',
        'gpu_platform_id': 0,
        'gpu_device_id': 0,  # Thêm dòng này để sử dụng GPU
             }

    lgb_model_v2 = lgb.train(
        params, lgb_train,
        num_boost_round=500,
        valid_sets=[lgb_val],
        callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(50)],
    )

    with open(MOD_DIR / 'lgbm_ranker_v2.pkl', 'wb') as f:
        pickle.dump(lgb_model_v2, f)
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Model v2 saved')
else:
    with open(MOD_DIR / 'lgbm_ranker_v2.pkl', 'rb') as f:
        lgb_model_v2 = pickle.load(f)
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Model v2 loaded from disk')

KeyboardInterrupt: 

In [ ]:
model_path = MOD_DIR / 'lgbm_ranker_v2.pkl'
if model_path.exists():
    print(f'File {model_path} exists.')
else:
    print(f'File {model_path} does NOT exist.')

File /content/drive/MyDrive/claude274v2/outputs/models/lgbm_ranker_v2.pkl exists.


In [ ]:
import pickle
with open(MOD_DIR / 'lgbm_ranker_v2.pkl', 'rb') as f:
    lgb_model_v2 = pickle.load(f)
print(f'[{datetime.now().strftime("%H:%M:%S")}] Model v2 loaded from disk')

[01:51:29] Model v2 loaded from disk


In [ ]:
# Kiểm tra label distribution của val
val_sample = pl.read_parquet(V2_DIR / 'val_feat_0000.parquet')
print(val_sample['label'].value_counts())
print(f'Positive rate: {val_sample["label"].mean():.4f}')

# Kiểm tra features có gì bất thường không
print(val_sample.select(FEATURE_COLS_V2).describe())

shape: (2, 2)
┌───────┬──────────┐
│ label ┆ count    │
│ ---   ┆ ---      │
│ i8    ┆ u32      │
╞═══════╪══════════╡
│ 0     ┆ 13714742 │
│ 1     ┆ 3414     │
└───────┴──────────┘
Positive rate: 0.0002
shape: (9, 29)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ age       ┆ is_club_m ┆ fashion_n ┆ … ┆ candidate ┆ candidate ┆ candidate ┆ candidat │
│ ---       ┆ ---       ┆ ember     ┆ ews_freq  ┆   ┆ _source_a ┆ _source_a ┆ _source_e ┆ e_source │
│ str       ┆ f64       ┆ ---       ┆ ---       ┆   ┆ ge_popula ┆ ls        ┆ mbedding_ ┆ _item_it │
│           ┆           ┆ f64       ┆ f64       ┆   ┆ r         ┆ ---       ┆ sim       ┆ em       │
│           ┆           ┆           ┆           ┆   ┆ ---       ┆ f64       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ f64       ┆           ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═════════

In [ ]:
# STEP 6: Feature importance
importance = sorted(zip(FEATURE_COLS_V2, lgb_model_v2.feature_importance('gain')), key=lambda x: -x[1])
print('Top-20 features by gain:')
for fname, score in importance[:20]:
    print(f'  {fname:<50s} {score:.1f}')

Top-20 features by gain:
  candidate_source_repurchase                        166744.0
  user_item_decay_weight                             343.9
  n_unique_articles_30d                              49.2
  n_purchases_14d                                    47.7
  article_popularity_14d                             42.8
  age                                                42.0
  product_type_no                                    35.6
  avg_price_30d                                      34.8
  embedding_similarity                               34.7
  article_popularity_age_group_14d                   28.0
  colour_group_code                                  19.6
  n_purchases_30d                                    18.3
  department_no                                      17.8
  days_since_last_purchase                           15.7
  candidate_source_item_item                         14.9
  days_since_article_last_sold                       10.5
  fashion_news_freq                       

## Section 5 — Prediction & Submission V2

In [ ]:
# STEP 1: Build test features — dùng all_candidates_v2
all_cands_v2 = pl.read_parquet(CAND_DIR / 'all_candidates_v2.parquet')
pred_cids    = all_cands_v2['customer_id'].unique().to_list()
pred_chunks  = [pred_cids[i:i+100_000] for i in range(0, len(pred_cids), 100_000)]

for i, cid_chunk in enumerate(pred_chunks):
    pp = V2_DIR / f'pred_feat_{i:04d}.parquet'
    if pp.exists():
        print(f'  Pred chunk {i+1}/{len(pred_chunks)} exists — skip')
        continue
    cid_set = set(cid_chunk)
    cands_c = all_cands_v2.filter(pl.col('customer_id').is_in(cid_set))
    feat_c  = build_features_v2(cands_c, label_tx=None)
    feat_c.write_parquet(pp)
    del cands_c, feat_c; gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Pred chunk {i+1}/{len(pred_chunks)} done')

del all_cands_v2; gc.collect()
pl.scan_parquet(V2_DIR / 'pred_feat_*.parquet').sink_parquet(V2_DIR / 'test_features_v2.parquet')
print(f'[{datetime.now().strftime("%H:%M:%S")}] Test features saved')

  Pred chunk 1/14 exists — skip
  Pred chunk 2/14 exists — skip
  Pred chunk 3/14 exists — skip
  Pred chunk 4/14 exists — skip
  Pred chunk 5/14 exists — skip
  Pred chunk 6/14 exists — skip
  Pred chunk 7/14 exists — skip
  Pred chunk 8/14 exists — skip
  Pred chunk 9/14 exists — skip
  Pred chunk 10/14 exists — skip
  Pred chunk 11/14 exists — skip
  Pred chunk 12/14 exists — skip
  Pred chunk 13/14 exists — skip
  Pred chunk 14/14 exists — skip
[01:18:49] Test features saved


In [ ]:
# STEP 2: Score theo chunk — chỉ giữ 3 cột để tránh OOM
import polars as pl
import gc
from datetime import datetime

FEATURE_COLS_V2 = [
    'age', 'is_club_member', 'fashion_news_freq',
    'n_purchases_7d', 'n_purchases_14d', 'n_purchases_30d',
    'n_unique_articles_30d', 'avg_price_30d', 'days_since_last_purchase',
    'article_popularity_7d', 'article_popularity_14d', 'article_popularity_age_group_14d',
    'days_since_article_last_sold',
    'product_type_no', 'colour_group_code', 'department_no',
    'user_has_bought_before', 'user_bought_same_product_type', 'user_bought_same_department',
    'embedding_similarity',
    'user_item_decay_weight',
    'sales_trend',
    'candidate_source_repurchase', 'candidate_source_global_popular',
    'candidate_source_age_popular', 'candidate_source_als',
    'candidate_source_embedding_sim', 'candidate_source_item_item',
]

all_cids = (
    pl.scan_parquet(V2_DIR / 'test_features_v2.parquet')
    .select('customer_id').unique().collect()
    ['customer_id'].to_list()
)
CHUNK_SIZE    = 100_000
chunks        = [all_cids[i:i+CHUNK_SIZE] for i in range(0, len(all_cids), CHUNK_SIZE)]
scored_paths  = []

for i, cid_chunk in enumerate(chunks):
    sp = V2_DIR / f'scored_{i:04d}.parquet'
    if sp.exists():
        scored_paths.append(sp)
        continue
    cid_set = set(cid_chunk)
    chunk = (
    pl.scan_parquet(V2_DIR / 'test_features_v2.parquet')
    .filter(pl.col('customer_id').is_in(cid_set))
    .collect()
)
    FCOLS   = [c for c in FEATURE_COLS_V2 if c in chunk.columns]
    scores  = lgb_model_v2.predict(chunk.select(FCOLS).to_pandas())
    scored  = chunk.select(['customer_id', 'article_id']).with_columns(pl.Series('score', scores))
    scored.write_parquet(sp)
    scored_paths.append(sp)
    del chunk, scores, scored; gc.collect()
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Score chunk {i+1}/{len(chunks)} done')




[01:52:36] Score chunk 1/14 done
[01:52:59] Score chunk 2/14 done
[01:53:21] Score chunk 3/14 done
[01:53:45] Score chunk 4/14 done
[01:54:23] Score chunk 5/14 done
[01:54:50] Score chunk 6/14 done
[01:55:14] Score chunk 7/14 done
[01:55:36] Score chunk 8/14 done
[01:55:58] Score chunk 9/14 done
[01:56:20] Score chunk 10/14 done
[01:56:59] Score chunk 11/14 done
[01:57:21] Score chunk 12/14 done
[01:57:43] Score chunk 13/14 done
[01:58:02] Score chunk 14/14 done


In [ ]:
# STEP 3: Merge scores và rank
scored_files = sorted(V2_DIR.glob('scored_*.parquet'))

# Add a check here to see if scored_files is empty
if not scored_files:
    print(f"Warning: No 'scored_*.parquet' files found in {V2_DIR}. Please ensure the previous step (scoring) completed successfully.")
    # You might want to exit or raise an error here, or handle the case appropriately.
    # For now, I will proceed, but expect the ValueError if the list remains empty.

top50_paths  = []

# Bước 1: Lấy top-50 per customer từng chunk
for i, sp in enumerate(scored_files):
    tp = V2_DIR / f'top50_{i:04d}.parquet'
    if not tp.exists():
        chunk = pl.read_parquet(sp)
        top50 = (
            chunk
            .sort(['customer_id', 'score'], descending=[False, True])
            .group_by('customer_id', maintain_order=True)
            .agg(pl.col('article_id').head(50).alias('top50'),
                 pl.col('score').head(50).alias('scores'))
        )
        top50.write_parquet(tp)
        del chunk, top50; gc.collect()
    top50_paths.append(tp)
    print(f'  {i+1}/{len(scored_files)} done')

# Bước 2: Concat top50 chunks — nhỏ vì mỗi customer chỉ còn 50 rows
print('Merging top50 chunks...')
merged = pl.concat([pl.read_parquet(p) for p in top50_paths])

# Bước 3: Explode, sort lại, lấy top-12 cuối
ranked = (
    merged
    .explode(['top50', 'scores'])
    .rename({'top50': 'article_id', 'scores': 'score'})
    .sort(['customer_id', 'score'], descending=[False, True])
    .group_by('customer_id', maintain_order=True)
    .agg(pl.col('article_id').head(12).alias('top12'))
)

for p in top50_paths: p.unlink()
del merged; gc.collect()
print(f'Ranked: {ranked.shape}')

# Fallback
global_top12 = (
    train_tx_clean.filter(pl.col('t_dat') > DATE_7D)
    .group_by('article_id').len()
    .sort('len', descending=True)
    .head(12)['article_id'].to_list()
)

missing = [c for c in customers['customer_id'].to_list()
           if c not in set(ranked['customer_id'].to_list())]
print(f'Missing customers (fallback): {len(missing):,}')
if missing:
    ranked = pl.concat([ranked, pl.DataFrame({
        'customer_id': missing,
        'top12': [global_top12] * len(missing)
    })])

submission = (
    customers.select('customer_id')
    .join(
        ranked.with_columns(pl.col('top12').list.join(' ').alias('prediction'))
              .select(['customer_id', 'prediction']),
        on='customer_id', how='left'
    )
    .with_columns(pl.col('prediction').fill_null(' '.join(global_top12)))
)

assert submission.shape[0] == customers.shape[0]
submission.write_csv(SUB_DIR / 'submission_v2.csv')
print(f'[{datetime.now().strftime("%H:%M:%S")}] Submission v2 saved: {submission.shape}')
print(submission.head(3))

  1/14 done
  2/14 done
  3/14 done
  4/14 done
  5/14 done
  6/14 done
  7/14 done
  8/14 done
  9/14 done
  10/14 done
  11/14 done
  12/14 done
  13/14 done
  14/14 done
Merging top50 chunks...
Ranked: (1371980, 2)


KeyboardInterrupt: 

In [ ]:
ranked.write_parquet(V2_DIR / 'ranked_v2.parquet')
print('ranked saved')

ranked saved


In [ ]:
ranked_set = set(ranked['customer_id'].to_list())
total      = customers.shape[0]
n_missing  = total - len(ranked_set)
print(f'Total: {total:,}, Ranked: {len(ranked_set):,}, Missing: {n_missing:,}')

Total: 1,371,980, Ranked: 1,371,980, Missing: 0


In [ ]:
submission = (
    ranked
    .with_columns(pl.col('top12').list.join(' ').alias('prediction'))
    .select(['customer_id', 'prediction'])
)

submission.write_csv(SUB_DIR / 'submission_v2.csv')
print(f'[{datetime.now().strftime("%H:%M:%S")}] Submission saved: {submission.shape}')
print(submission.head(3))

[03:39:36] Submission saved: (1371980, 2)
shape: (3, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ customer_id                     ┆ prediction                      │
│ ---                             ┆ ---                             │
│ str                             ┆ str                             │
╞═════════════════════════════════╪═════════════════════════════════╡
│ 00000dbacae5abe5e23885899a1fa4… ┆ 0568601043 0896169002 08323070… │
│ 0000423b00ade91418cceaf3b26c6a… ┆ 0903926001 0762846006 09366220… │
│ 000058a12d5b43e67d225668fa1f8d… ┆ 0794321007 0912204001 08819420… │
└─────────────────────────────────┴─────────────────────────────────┘


## Section 6 — Đánh giá MAP@12

In [ ]:
# Tính MAP@12 trên tuần test thật
def apk(actual, predicted, k=12):
    if not actual: return 0.0
    predicted = predicted[:k]
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p in actual:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)

def mapk(actuals, predicteds, k=12):
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, predicteds)]))

# Ground truth từ tuần test
test_tx = transactions.filter(pl.col('t_dat') >= TEST_START_DATE)
ground_truth = (
    test_tx.group_by('customer_id')
    .agg(pl.col('article_id').unique().alias('bought'))
)
print(f'Customers who bought in test week: {ground_truth.shape[0]:,}')

# Load submission v2
sub_v2 = pl.read_csv(SUB_DIR / 'submission_v2.csv')
eval_df = ground_truth.join(
    sub_v2.with_columns(pl.col('prediction').str.split(' ').alias('top12'))
          .select(['customer_id', 'top12']),
    on='customer_id', how='inner'
)

map12_v2 = mapk(eval_df['bought'].to_list(), eval_df['top12'].to_list())
print(f'\n=== KẾT QUẢ ===')
print(f'MAP@12 V2: {map12_v2:.4f}')

# So sánh với v1 nếu có
sub_v1_path = SUB_DIR / 'submission.csv'
if sub_v1_path.exists():
    sub_v1 = pl.read_csv(sub_v1_path)
    eval_v1 = ground_truth.join(
        sub_v1.with_columns(pl.col('prediction').str.split(' ').alias('top12'))
              .select(['customer_id', 'top12']),
        on='customer_id', how='inner'
    )
    map12_v1 = mapk(eval_v1['bought'].to_list(), eval_v1['top12'].to_list())
    print(f'MAP@12 V1: {map12_v1:.4f}')
    print(f'Improvement: {map12_v2 - map12_v1:+.4f}')

Customers who bought in test week: 68,984

=== KẾT QUẢ ===
MAP@12 V2: 0.0273


In [ ]:
# Ablation study — tắt từng source, đo MAP@12
# Dùng scored files đã có để không cần rebuild features

sources = ['repurchase', 'global_popular', 'age_popular', 'als', 'embedding_sim', 'item_item']
ablation_results = [{'Removed': 'None (full)', 'MAP@12': map12_v2, 'Delta': 0.0}]

# Prepare necessary data once
# ground_truth, global_top12, and map12_v2 are assumed to be available from previous cells.

# Define global_top12 for fallback in this context
global_top12 = (
    train_tx.filter(pl.col('t_dat') > DATE_7D)
    .group_by('article_id').len()
    .sort('len', descending=True)
    .head(12)['article_id'].to_list()
)

all_customer_ids = customers['customer_id'].to_list() # All customer IDs for fallback handling
scored_files = sorted(V2_DIR.glob('scored_*.parquet')) # List of scored files, which are customer-chunked

for src in sources:
    print(f'Processing ablation for: - {src}')
    current_src_ranked_chunks = []

    for i, scored_file_path in enumerate(scored_files):
        # Load scored candidates for the current customer chunk
        scored_chunk = pl.read_parquet(scored_file_path)

        # Get customer IDs present in this chunk
        chunk_customer_ids = scored_chunk['customer_id'].unique().to_list()

        # Lazily load and filter all_candidates_v2 for the current chunk of customers
        # and exclude the source being ablated. Then collect only necessary columns.
        ablated_cands_chunk = (
            pl.scan_parquet(CAND_DIR / 'all_candidates_v2.parquet')
            .filter(pl.col('customer_id').is_in(chunk_customer_ids))
            .filter(pl.col('source') != src)
            .select(['customer_id', 'article_id']) # Only select necessary columns for the join
            .collect() # Collect the valid pairs for the current chunk and source
        )

        # Join the scored chunk with the ablated candidates for these customers
        scored_ablated_chunk = scored_chunk.join(
            ablated_cands_chunk,
            on=['customer_id', 'article_id'],
            how='inner'
        )

        # Rank candidates within the current chunk
        ranked_abl_chunk = (
            scored_ablated_chunk
            .sort(['customer_id', 'score'], descending=[False, True])
            .group_by('customer_id', maintain_order=True)
            .agg(pl.col('article_id').head(12).alias('top12'))
        )
        current_src_ranked_chunks.append(ranked_abl_chunk)

        # Explicitly delete variables and collect garbage to free memory
        del scored_chunk, ablated_cands_chunk, scored_ablated_chunk, ranked_abl_chunk; gc.collect()

        if (i + 1) % 2 == 0: # Print progress for every few chunks
            print(f'  Chunk {i+1}/{len(scored_files)} processed for source: {src}')

    # After processing all chunks for the current source, concatenate them
    ranked_abl = pl.concat(current_src_ranked_chunks)
    del current_src_ranked_chunks; gc.collect()

    # Handle missing customers for the full set of customers
    # Convert to set only once for efficiency
    ranked_abl_customer_set = set(ranked_abl['customer_id'].to_list())
    missing_cids = [c for c in all_customer_ids if c not in ranked_abl_customer_set]
    if missing_cids:
        ranked_abl = pl.concat([ranked_abl, pl.DataFrame({
            'customer_id': missing_cids, 'top12': [global_top12] * len(missing_cids)
        })])

    # Evaluate MAP@12
    eval_abl = ground_truth.join(ranked_abl.select(['customer_id', 'top12']),
                                  on='customer_id', how='inner')
    m     = mapk(eval_abl['bought'].to_list(), eval_abl['top12'].to_list())
    delta = m - map12_v2
    ablation_results.append({'Removed': f'- {src}', 'MAP@12': m, 'Delta': delta})
    print(f'- {src:<20s}  MAP@12={m:.4f}  delta={delta:+.4f}')

    # Clean up large intermediate results for the current source
    del ranked_abl, eval_abl, ranked_abl_customer_set; gc.collect()

ablation_df = pl.DataFrame(ablation_results)
ablation_df.write_csv(OUT_DIR / 'ablation_v2.csv')
print('\n=== ABLATION TABLE ===')
print(ablation_df)

Processing ablation for: - repurchase
  Chunk 2/14 processed for source: repurchase
  Chunk 4/14 processed for source: repurchase
  Chunk 6/14 processed for source: repurchase
  Chunk 8/14 processed for source: repurchase
  Chunk 10/14 processed for source: repurchase
  Chunk 12/14 processed for source: repurchase
  Chunk 14/14 processed for source: repurchase
- repurchase            MAP@12=0.0242  delta=-0.0032
Processing ablation for: - global_popular
  Chunk 2/14 processed for source: global_popular
  Chunk 4/14 processed for source: global_popular
  Chunk 6/14 processed for source: global_popular
  Chunk 8/14 processed for source: global_popular
  Chunk 10/14 processed for source: global_popular
  Chunk 12/14 processed for source: global_popular
  Chunk 14/14 processed for source: global_popular
- global_popular        MAP@12=0.0353  delta=+0.0080
Processing ablation for: - age_popular
  Chunk 2/14 processed for source: age_popular
  Chunk 4/14 processed for source: age_popular
  C

In [ ]:
import json

nb_path = '/content/your_notebook.ipynb'  # đổi tên file

with open(nb_path, 'r') as f:
    nb = json.load(f)

# Xóa hoàn toàn widgets khỏi metadata
if 'widgets' in nb['metadata']:
    del nb['metadata']['widgets']
    print("Đã xóa widgets metadata")
else:
    print("Không tìm thấy widgets")

# Xóa referenced_widgets trong từng cell
for cell in nb['cells']:
    if 'metadata' in cell:
        if 'referenced_widgets' in cell['metadata']:
            del cell['metadata']['referenced_widgets']

with open(nb_path, 'w') as f:
    json.dump(nb, f, indent=1)

print("Saved! Mở lại notebook xem sao.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/your_notebook.ipynb'